# Notebook 4 — Modelling

I chose three modelling pieces, each chosen to test one part of the blog's argument:

| Question | Method | Output |
|----------|--------|--------|
| Does Serie A's foreign-player share predict Italy's tournament performance? | OLS with HC1 robust SE | `model1_summary.txt` |
| Same question for total squad value (placebo for "money matters") | OLS | `model2_summary.txt` |
| Does the *gap* between Serie A's foreign % and peer leagues' average predict performance? | OLS | `model3_summary.txt` |

**It is important to understant the dataset is extremely small (≈ 12 tournament observations, 2006–2024), so every estimate carries very wide uncertainty. The point is a disciplined, replicable analysis, not a causal claim.**

I use OLS rather than logistic regression for qualification because there are only two failures (n_failures = 2 ≪ n_features) so a logit would be overfit. I instead model the continuous `performance_score` (1–7). This again limits the reliability of the regression.

I also include a panel regression across the 10 leagues to test whether the foreign-player share explains the league's *competitive balance* (Gini), which is the structural mechanism the blog discusses.


## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

CLEAN_DIR = Path('../data/clean')
OUT_DIR   = Path('../output')
OUT_DIR.mkdir(parents=True, exist_ok=True)

italy_final   = pd.read_csv(CLEAN_DIR / 'italy_final.csv')
league_season = pd.read_csv(CLEAN_DIR / 'league_season.csv')

# Tournament summer 2006 is Serie A 2005/6, so start year 2005, therefore not included.
italy_final['serie_a_season_start'] = italy_final['year'] - 1

serie_a = league_season[league_season.country == 'Italy'].copy()
peers   = league_season[league_season.country != 'Italy'].copy()

peer_avg = (peers.groupby('season_start')
                  .agg(peer_pct_foreign=('pct_foreign', 'mean'),
                       peer_gini=('gini_coefficient','mean'),
                       peer_value_m=('total_market_value_m','mean'))
                  .reset_index())

df = (italy_final
      .merge(serie_a[['season_start','pct_foreign','gini_coefficient','total_market_value_m']]
                .rename(columns={'pct_foreign':'serie_a_pct_foreign',
                                 'gini_coefficient':'serie_a_gini',
                                 'total_market_value_m':'serie_a_value_m'}),
             left_on='serie_a_season_start', right_on='season_start', how='left')
      .merge(peer_avg, on='season_start', how='left')
      .drop(columns=['serie_a_season_start','season_start']))

# Foreign-player gap: how much more foreign Serie A is than the peer-league average
df['foreign_gap'] = df['serie_a_pct_foreign'] - df['peer_pct_foreign']

# Drop rows where Serie A foreign % is missing (would only happen if season missing)
df = df.dropna(subset=['serie_a_pct_foreign','performance_score']).reset_index(drop=True)

print(f'Modelling dataset: {len(df)} tournaments')
print(df[['year','tournament','result','performance_score',
          'serie_a_pct_foreign','peer_pct_foreign','foreign_gap',
          'serie_a_gini','total_value_m']].to_string(index=False))

Modelling dataset: 10 tournaments
 year tournament          result  performance_score  serie_a_pct_foreign  peer_pct_foreign  foreign_gap  serie_a_gini  total_value_m
 2008       Euro   Quarter-final                  4                32.93         43.527778   -10.597778        0.3833         421.60
 2010  World Cup     Group Stage                  2                37.82         44.107778    -6.287778        0.3621         480.65
 2012       Euro       Runner-up                  6                43.75         44.398889    -0.648889        0.3756         470.90
 2014  World Cup     Group Stage                  2                46.09         46.101111    -0.011111        0.3343         507.80
 2016       Euro   Quarter-final                  4                50.92         47.715556     3.204444        0.3926         485.30
 2018  World Cup Did Not Qualify                  1                50.48         50.338889     0.141111        0.4265         453.50
 2021       Euro          Winner   

---
## Model 1 — Italy's tournament performance vs Serie A foreign-player share

A reminder of the believed hypothesis: as Serie A fills with foreigners, fewer Italian players get playing minutes, and the national team performs worse. We expect a **negative** coefficient on `serie_a_pct_foreign`.

Specification: `performance_score = β0 + β1 * serie_a_pct_foreign + ε` with HC1 robust standard errors (small sample).


In [20]:
m1 = smf.ols('performance_score ~ serie_a_pct_foreign', data=df).fit(cov_type='HC1')
print(m1.summary())

with open(OUT_DIR / 'model1_summary.txt','w') as f:
    f.write(str(m1.summary()))
print(f'\nSaved {OUT_DIR/"model1_summary.txt"}')

                            OLS Regression Results                            
Dep. Variable:      performance_score   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                 -0.123
Method:                 Least Squares   F-statistic:                   0.04099
Date:                Wed, 22 Apr 2026   Prob (F-statistic):              0.845
Time:                        19:26:38   Log-Likelihood:                -20.599
No. Observations:                  10   AIC:                             45.20
Df Residuals:                       8   BIC:                             45.80
Df Model:                           1                                         
Covariance Type:                  HC1                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept               3.7102    

---
## Model 2 — performance vs Italy's own squad market value

If "Italy is just declining because it has weaker players" then the team's own market value should predict performance. If this coefficient is large and significant, but the foreign-player coefficient (Model 1) isn't, then the structural-league explanation is weaker than the simpler "weaker squad" one.

Specification: `performance_score = β0 + β1 * total_value_m + ε`.


In [21]:
m2 = smf.ols('performance_score ~ total_value_m', data=df).fit(cov_type='HC1')
print(m2.summary())

with open(OUT_DIR / 'model2_summary.txt','w') as f:
    f.write(str(m2.summary()))
print(f'\nSaved {OUT_DIR/"model2_summary.txt"}')

                            OLS Regression Results                            
Dep. Variable:      performance_score   R-squared:                       0.184
Model:                            OLS   Adj. R-squared:                  0.068
Method:                 Least Squares   F-statistic:                     1.718
Date:                Wed, 22 Apr 2026   Prob (F-statistic):              0.231
Time:                        19:26:38   Log-Likelihood:                -18.093
No. Observations:                   9   AIC:                             40.19
Df Residuals:                       7   BIC:                             40.58
Df Model:                           1                                         
Covariance Type:                  HC1                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept         1.3244      1.686      0.786

---
## Model 3 — Performance vs the *gap* between Serie A and peer leagues

A relative measure: how much more foreign-heavy Serie A is than the average top-10 league. If Serie A's openness is a problem, what should bite is its **outlier** status, not its absolute level. This was briefly shown in the third notebook.

Specification: `performance_score = β0 + β1 * foreign_gap + ε`.


In [22]:
m3 = smf.ols('performance_score ~ foreign_gap', data=df).fit(cov_type='HC1')
print(m3.summary())

with open(OUT_DIR / 'model3_summary.txt','w') as f:
    f.write(str(m3.summary()))
print(f'\nSaved {OUT_DIR/"model3_summary.txt"}')

                            OLS Regression Results                            
Dep. Variable:      performance_score   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.125
Method:                 Least Squares   F-statistic:                  0.003370
Date:                Wed, 22 Apr 2026   Prob (F-statistic):              0.955
Time:                        19:26:38   Log-Likelihood:                -20.607
No. Observations:                  10   AIC:                             45.21
Df Residuals:                       8   BIC:                             45.82
Df Model:                           1                                         
Covariance Type:                  HC1                                         
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       3.2932      0.660      4.986      

---
## Save the modelling dataset

So that a reader replicating from the cleaned CSVs can recover the exact rows fed to the regressions.

In [23]:
out = CLEAN_DIR / 'modelling_dataset.csv'
df.to_csv(out, index=False)
print(f'Saved {out} ({len(df)} rows, {len(df.columns)} columns)')
print(f'Columns: {list(df.columns)}')

Saved ../data/clean/modelling_dataset.csv (10 rows, 18 columns)
Columns: ['year', 'tournament', 'result', 'performance_score', 'qualified', 'host', 'players', 'avg_value_m', 'total_value_m', 'top_value_m', 'avg_top11_value_m', 'serie_a_pct_foreign', 'serie_a_gini', 'serie_a_value_m', 'peer_pct_foreign', 'peer_gini', 'peer_value_m', 'foreign_gap']


---
## What the models actually say
In the blog post I will explain:
1. **Sign and significance** of `serie_a_pct_foreign` in Model 1 — does Italy do worse when Serie A is more foreign?
2. **Comparison** between Models 1 and 2 — is the foreign-share story stronger or weaker than the brute "richer squad wins" story?
3. **The panel regression** — is the foreign-share / Gini link a Serie A peculiarity or a general top-flight phenomenon?

Again, it is important to understand every result is **not** proof of causal effect.
